In [ ]:
import glob
import numpy as np
import pandas as pd
import xarray as xr

swot_dir = '/data1/data/obs/level3/SWOT/V-3/'
coord_grids = np.load("input_data/coord_grids.npy")   # grille GLOBALE de NeurOST
print(coord_grids.shape)        # à regarder : (nb_patchs, 128, 128, 2) ?

# --- boîte (lon/lat min/max) de chaque patch ---
# TODO 1 : vérifier dans le print ci-dessus quel indice est lon et lequel est lat
lon_patch = coord_grids[:, :, :, 0] % 360
lat_patch = coord_grids[:, :, :, 1]

lon_min_p = lon_patch.min(axis=(1, 2))
lon_max_p = lon_patch.max(axis=(1, 2))
lat_min_p = lat_patch.min(axis=(1, 2))
lat_max_p = lat_patch.max(axis=(1, 2))

n_min = 500     # nb minimum de points KaRIn (à 2 km) pour que le patch compte comme exemple

# quelques jours de train pour tester (pas les 327)
jours_test = pd.to_datetime(["2023-08-15", "2023-10-01", "2024-04-15"])

for jour in jours_test:
    fichiers = sorted(glob.glob(f"{swot_dir}/*_{jour:%Y%m%d}T*.nc"))
    n_points = np.zeros(len(coord_grids))     # nb de points KaRIn dans chaque patch

    for f in fichiers:
        ds = xr.open_dataset(f)
        lon = ds.longitude.values.ravel() % 360
        lat = ds.latitude.values.ravel()
        ssh = ds.ssha_unfiltered.values.ravel()
        ok = np.isfinite(ssh)
        lon, lat = lon[ok], lat[ok]
        ds.close()

        for p in range(len(coord_grids)):
            # TODO 2 : masque des points dans la boîte du patch p
            #          (lon entre lon_min_p[p] et lon_max_p[p], lat entre ...)
            dedans = (lon >= lon_min_p[p]) & (lon <= lon_max_p[p]) & (lat >= lat_min_p[p]) & (lat <= lat_max_p[p])
            n_points[p] += dedans.sum()

    # TODO 3 : nombre de patchs avec au moins n_min points
    n_exemples = (n_points >= n_min).sum()
    print(jour.date(), len(fichiers), "fichiers SWOT ->", n_exemples, "patchs utilisables")